# Setup

## Imports and configuration

In [1]:
import sys
import os
current_dir = os.getcwd()
project_root = os.path.abspath(os.path.join(current_dir, '..'))
if project_root not in sys.path:
    sys.path.append(project_root)
os.environ["LIBGL_ALWAYS_SOFTWARE"] = "1"
import glob
import tempfile
import hashlib
import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
from tqdm import tqdm
from src.extractors.visual.visual_feature_extractor import VisualFeatureExtractor
from huggingface_hub import snapshot_download
from scenedetect import detect, ContentDetector
from src.common.constants import OPENFACE_BIN
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 5)

# Hardware Configuration
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Directory & Binary Path Configurations
DATA_DIR = os.path.expanduser("../data")
HF_DATASET_NAME = "dandyidk/Multi-Modal-Lie-Detection"
print(f"CUDA Available: {torch.cuda.is_available()} | Device: {DEVICE}")
print(f"Loading Hugging Face Dataset: {HF_DATASET_NAME}")
print(f"OpenFace Executable: {OPENFACE_BIN}")

CUDA Available: True | Device: cuda
Loading Hugging Face Dataset: dandyidk/Multi-Modal-Lie-Detection
OpenFace Executable: /home/dandy/OpenFace/build/bin/FeatureExtraction


## Remote Dataset Inspection

In [2]:
from huggingface_hub import HfApi

api = HfApi()

files = api.list_repo_files(
    repo_id="dandyidk/Multimodal",
    repo_type="dataset",
)

# Use a tuple of folder strings to check multiple paths
target_folders = ("deception", "truth") 

filtered_files = []
for filename in files:
    if filename.startswith(target_folders):
        filtered_files.append(filename)
        # pass 

print(f"Found {len(filtered_files)} files across the target folders.")

Found 1700 files across the target folders.


### Dataset Download and Metadata Extraction
Download the dataset using `snapshot_download`. Once downloaded, recursively locate all `.mp4` files and use OpenCV (`cv2`) to extract baseline video metadata, such as frame count, FPS, and total duration in seconds. This information is saved into a Pandas DataFrame (`meta_df`) for downstream processing.

In [3]:
os.makedirs(DATA_DIR, exist_ok=True)
repo_dir = snapshot_download(
    repo_id=HF_DATASET_NAME,
    repo_type="dataset",
    local_dir=DATA_DIR
)

# Locate all local MP4 files recursively
video_paths = sorted(glob.glob(os.path.join(DATA_DIR, "**", "*.mp4"), recursive=True))

video_stats = []

for idx, video_path in enumerate(video_paths):
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        print(f"Warning: Failed to open video at {video_path}")
        continue

    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    fps = cap.get(cv2.CAP_PROP_FPS)
    duration = frame_count / fps if fps and fps > 0 else 0.0

    video_stats.append({
        "sample_index": idx,
        "filename": os.path.basename(video_path),
        "path": video_path,  # Physical disk path for OpenFace
        "frame_count": frame_count,
        "fps": round(float(fps), 2),
        "duration_sec": round(float(duration), 2),
    })
    cap.release()

meta_df = pd.DataFrame(video_stats)
print(f"Successfully loaded and verified {len(meta_df)} video files.")
meta_df.head()

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 1717 files:   0%|          | 0/1717 [00:00<?, ?it/s]

Successfully loaded and verified 1579 video files.


,sample_index,filename,path,frame_count,fps,duration_sec
0,0,LS_WILTY_EP72_lie14.mp4,../data/deception/DOLOS/Female/Catherine/LS_WI...,175,25.0,7.0
1,1,LS_WILTY_EP72_lie15.mp4,../data/deception/DOLOS/Female/Catherine/LS_WI...,125,25.0,5.0
2,2,LS_WILTY_EP72_lie16.mp4,../data/deception/DOLOS/Female/Catherine/LS_WI...,75,25.0,3.0
3,3,LS_WILTY_EP72_lie17.mp4,../data/deception/DOLOS/Female/Catherine/LS_WI...,175,25.0,7.0
4,4,AN_WILTY_EP19_lie1.mp4,../data/deception/DOLOS/Female/Christine Bleak...,144,24.0,6.0


In [4]:
"""Extract frame-level OpenFace features and Gaze-LLE gaze-target features."""

import logging
import os
import shutil
import subprocess
import tempfile
from typing import Dict, List, Optional, Sequence

import cv2
import numpy as np
import pandas as pd
import torch
from PIL import Image

from src.models.gazelle.gazelle import get_gazelle_model

logger = logging.getLogger(__name__)

GAZELLE_INPUT_SIZE = 448


class VisualFeatureExtractor:
    """Combine OpenFace AUs/pose/gaze with Gaze-LLE gaze-target features."""

    POSE = ["pose_Tx", "pose_Ty", "pose_Tz", "pose_Rx", "pose_Ry", "pose_Rz"]
    GAZE = [
        "gaze_0_x", "gaze_0_y", "gaze_0_z",
        "gaze_1_x", "gaze_1_y", "gaze_1_z",
        "gaze_angle_x", "gaze_angle_y",
    ]
    AU_INTENSITY = [
        "AU01_r", "AU02_r", "AU04_r", "AU05_r", "AU06_r", "AU07_r", "AU09_r",
        "AU10_r", "AU12_r", "AU14_r", "AU15_r", "AU17_r", "AU20_r", "AU23_r",
        "AU25_r", "AU26_r", "AU45_r",
    ]
    AU_PRESENCE = [
        "AU01_c", "AU02_c", "AU04_c", "AU05_c", "AU06_c", "AU07_c", "AU09_c",
        "AU10_c", "AU12_c", "AU14_c", "AU15_c", "AU17_c", "AU20_c", "AU23_c",
        "AU25_c", "AU26_c", "AU28_c", "AU45_c",
    ]
    OPENFACE_COLS = AU_INTENSITY + AU_PRESENCE + POSE + GAZE
    QUALITY_COLS = ["valid", "confidence"]
    LANDMARK_X = [f"x_{i}" for i in range(68)]
    LANDMARK_Y = [f"y_{i}" for i in range(68)]

    # Per-frame Gaze-LLE outputs (all in [0, 1], coords normalized to the frame)
    GAZELLE_COLS = [
        "tgt_x", "tgt_y",        # heatmap-weighted mean gaze target (soft-argmax)
        "peak_x", "peak_y",      # heatmap argmax
        "inout",                 # in-frame score (1 = target in frame)
        "peak_val",              # heatmap max value (confidence)
        "entropy",               # normalized heatmap entropy (diffuseness)
    ]

    def __init__(
        self,
        gazelle_checkpoint_path="path/to/gazelle_checkpoint.pt",
        gazelle_model_name="gazelle_dinov2_vitl14_inout",
        openface_bin_path="FeatureExtraction",
        device=None,
        min_confidence=0.75,
        bbox_margin=0.15,
        bbox_forehead=0.35,
        include_quality_flags=True,
        add_pose_delta=False,
        scale_features=True,
        batch_size=8,
        openface_timeout=300.0,
    ):
        self.device = device or ("cuda" if torch.cuda.is_available() else "cpu")
        self.min_confidence = min_confidence
        self.bbox_margin = bbox_margin
        self.bbox_forehead = bbox_forehead  # extra top expansion; landmarks stop at the brows
        self.include_quality_flags = include_quality_flags
        self.add_pose_delta = add_pose_delta
        self.scale_features = scale_features
        self.batch_size = batch_size
        self.openface_timeout = openface_timeout

        self.openface_bin = shutil.which(openface_bin_path)
        if not self.openface_bin:
            raise FileNotFoundError(f"OpenFace binary not found: {openface_bin_path!r}")

        logger.info("Loading Gaze-LLE (%s) from: %s", gazelle_model_name, gazelle_checkpoint_path)
        self.gazelle, self.transform = get_gazelle_model(gazelle_model_name)
        self.gazelle.load_gazelle_state_dict(
            torch.load(gazelle_checkpoint_path, weights_only=True)
        )
        self.gazelle = self.gazelle.to(self.device)
        self.gazelle.eval()

    @property
    def feature_names(self) -> List[str]:
        names = self.OPENFACE_COLS.copy()
        if self.include_quality_flags:
            names += self.QUALITY_COLS
        if self.add_pose_delta:
            names += [f"d_{c}" for c in self.POSE]
        return names + [f"gz_{c}" for c in self.GAZELLE_COLS]

    @property
    def feature_dim(self) -> int:
        return len(self.feature_names)

    def run_openface(self, video_path: str, output_dir: str) -> pd.DataFrame:
        """Run OpenFace (with 2D landmarks, needed for head boxes) and return its CSV."""
        cmd = [
            self.openface_bin, "-f", video_path, "-out_dir", output_dir,
            "-pose", "-gaze", "-aus", "-2Dfp", "-q",
        ]
        try:
            result = subprocess.run(
                cmd, capture_output=True, text=True, timeout=self.openface_timeout
            )
        except subprocess.TimeoutExpired as exc:
            raise RuntimeError(
                f"OpenFace timed out after {self.openface_timeout}s: {video_path}"
            ) from exc
        if result.returncode:
            raise RuntimeError(f"OpenFace failed: {result.stderr}")

        name = os.path.splitext(os.path.basename(video_path))[0]
        csv_path = os.path.join(output_dir, f"{name}.csv")
        if not os.path.isfile(csv_path):
            raise FileNotFoundError(f"OpenFace CSV not found: {csv_path}")

        df = pd.read_csv(csv_path)
        df.columns = df.columns.str.strip()
        return df

    def _validity(self, df: pd.DataFrame) -> np.ndarray:
        """Mark frames with a successful, sufficiently confident face track."""
        required = ["success", "confidence"]
        missing = [c for c in required if c not in df]
        if missing:
            raise KeyError(f"Missing OpenFace columns: {missing}")
        return (
            (df["success"].to_numpy(dtype=float) > 0.5)
            & (df["confidence"].to_numpy(dtype=float) >= self.min_confidence)
        )

    def _clean_openface(self, df: pd.DataFrame, valid: np.ndarray) -> np.ndarray:
        """Mask failed tracks, interpolate continuous features, and scale values."""
        missing = [c for c in self.OPENFACE_COLS if c not in df]
        if missing:
            raise KeyError(f"Missing OpenFace columns: {missing}")

        feats = df[self.OPENFACE_COLS].astype(float).copy()
        feats.loc[~valid] = np.nan

        if valid.any():
            continuous = self.AU_INTENSITY + self.POSE + self.GAZE
            feats[continuous] = feats[continuous].interpolate(limit_direction="both")
            feats[self.AU_PRESENCE] = feats[self.AU_PRESENCE].ffill().bfill()
        feats = feats.fillna(0.0)

        if self.scale_features:
            feats[self.AU_INTENSITY] /= 5.0
            feats[["pose_Tx", "pose_Ty", "pose_Tz"]] /= 1000.0
        return feats.to_numpy(dtype=np.float32)

    def _head_boxes(self, df: pd.DataFrame, valid: np.ndarray) -> Optional[np.ndarray]:
        """Pixel-space head boxes (x0, y0, x1, y1) from landmarks, reusing nearest valid boxes."""
        cols = self.LANDMARK_X + self.LANDMARK_Y
        if not all(c in df for c in cols) or not valid.any():
            return None

        xs, ys = df[self.LANDMARK_X].to_numpy(float), df[self.LANDMARK_Y].to_numpy(float)
        x0, x1, y0, y1 = xs.min(axis=1), xs.max(axis=1), ys.min(axis=1), ys.max(axis=1)
        bw, bh = x1 - x0, y1 - y0
        m = self.bbox_margin
        boxes = np.column_stack((
            x0 - m * bw,
            y0 - (m + self.bbox_forehead) * bh,
            x1 + m * bw,
            y1 + m * bh,
        ))
        boxes[~valid] = np.nan
        boxes = pd.DataFrame(boxes).ffill().bfill().to_numpy()
        return None if np.isnan(boxes).any() else boxes

    @staticmethod
    def _heatmap_stats(heatmap: torch.Tensor, inout: Optional[torch.Tensor]) -> torch.Tensor:
        """(B, H, W) heatmaps -> (B, len(GAZELLE_COLS)) summary features."""
        b, h, w = heatmap.shape
        hm = heatmap.float()
        p = hm / hm.sum(dim=(-1, -2), keepdim=True).clamp_min(1e-8)

        xs = (torch.arange(w, device=hm.device, dtype=torch.float32) + 0.5) / w
        ys = (torch.arange(h, device=hm.device, dtype=torch.float32) + 0.5) / h
        tgt_x = (p.sum(dim=1) * xs).sum(-1)
        tgt_y = (p.sum(dim=2) * ys).sum(-1)

        flat = hm.reshape(b, -1)
        peak_val, idx = flat.max(dim=1)
        peak_x = ((idx % w).float() + 0.5) / w
        peak_y = ((idx // w).float() + 0.5) / h

        entropy = -(p * torch.log(p + 1e-8)).sum(dim=(-1, -2)) / np.log(h * w)
        inout_v = torch.ones(b, device=hm.device) if inout is None else inout.float().reshape(b)

        return torch.stack(
            (tgt_x, tgt_y, peak_x, peak_y, inout_v, peak_val, entropy), dim=1
        )

    @torch.no_grad()
    def extract_gaze_features(
        self, video_path: str, frame_indices: Sequence[int],
        boxes: Optional[np.ndarray] = None,
    ) -> torch.Tensor:
        """Return Gaze-LLE features (n, len(GAZELLE_COLS)) for the requested zero-based frames.

        `boxes` are pixel-space head boxes aligned with `frame_indices`.
        Frames without a usable head box get all-zero features.
        """
        n_cols = len(self.GAZELLE_COLS)
        n = len(frame_indices)
        if not n:
            return torch.empty((0, n_cols))

        cap = cv2.VideoCapture(video_path)
        fw = cap.get(cv2.CAP_PROP_FRAME_WIDTH)
        fh = cap.get(cv2.CAP_PROP_FRAME_HEIGHT)

        # Normalize boxes to [0, 1] w.r.t. the full frame, as Gaze-LLE expects.
        norm_boxes = np.zeros((n, 4), dtype=np.float32)
        has_box = np.zeros(n, dtype=bool)
        if boxes is not None and fw > 0 and fh > 0:
            nb = np.clip(boxes / np.array([fw, fh, fw, fh]), 0.0, 1.0)
            has_box = (nb[:, 2] - nb[:, 0] > 0.01) & (nb[:, 3] - nb[:, 1] > 0.01)
            norm_boxes = nb.astype(np.float32)
        norm_boxes[~has_box] = (0.0, 0.0, 1.0, 1.0)  # dummy, outputs zeroed later

        # Keep duplicate frame requests mapped to all corresponding output positions.
        wanted: Dict[int, List[int]] = {}
        for pos, frame_idx in enumerate(frame_indices):
            wanted.setdefault(int(frame_idx), []).append(pos)

        # Resize at read time (uint8, 448x448) to avoid holding full-res frames in RAM.
        images: List[Optional[Image.Image]] = [None] * n
        idx, last = 0, max(wanted)
        while idx <= last and cap.grab():
            if idx in wanted:
                ok, frame = cap.retrieve()
                if ok:
                    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                    img = Image.fromarray(rgb).resize(
                        (GAZELLE_INPUT_SIZE, GAZELLE_INPUT_SIZE), Image.BICUBIC
                    )
                    for pos in wanted[idx]:
                        images[pos] = img
            idx += 1
        cap.release()

        # Fill unreadable frames from the nearest readable neighbour.
        if any(im is None for im in images):
            logger.warning("%s: some frames could not be read; filling from neighbours", video_path)
            previous = None
            for i, im in enumerate(images):
                if im is None:
                    images[i] = previous
                else:
                    previous = im
            following = None
            for i in reversed(range(n)):
                if images[i] is None:
                    images[i] = following
                else:
                    following = images[i]
            blank = Image.fromarray(
                np.zeros((GAZELLE_INPUT_SIZE, GAZELLE_INPUT_SIZE, 3), dtype=np.uint8)
            )
            images = [blank if im is None else im for im in images]

        use_amp = self.device.startswith("cuda")
        outputs = []
        for start in range(0, n, self.batch_size):
            sl = slice(start, start + self.batch_size)
            batch_tensor = torch.stack([self.transform(im) for im in images[sl]]).to(self.device)
            batch_boxes = [[tuple(float(v) for v in b)] for b in norm_boxes[sl]]

            with torch.autocast("cuda", dtype=torch.float16, enabled=use_amp):
                out = self.gazelle({"images": batch_tensor, "bboxes": batch_boxes})

            # One person per image -> take person 0 from each per-image list.
            heatmap = torch.stack([h[0] for h in out["heatmap"]])
            inout = None
            if out.get("inout") is not None:
                inout = torch.stack([s[0] for s in out["inout"]])
            outputs.append(self._heatmap_stats(heatmap, inout).cpu())

        feats = torch.cat(outputs)
        feats[torch.from_numpy(~has_box)] = 0.0
        return feats

    @staticmethod
    def _video_frame_count(video_path: str) -> int:
        cap = cv2.VideoCapture(video_path)
        count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        cap.release()
        return max(count, 0)

    def extract_features(self, video_path: str, target_t: int = 90) -> torch.FloatTensor:
        """Return visual features with shape (1, target_t, feature_dim)."""
        with tempfile.TemporaryDirectory() as tmp:
            df = self.run_openface(video_path, tmp)

        if len(df):
            valid = self._validity(df)
            if not valid.any():
                logger.warning("%s: no valid face tracks; OpenFace features are zero", video_path)

            cleaned = self._clean_openface(df, valid)
            rows = np.linspace(0, len(df) - 1, target_t).round().astype(int)
            of_feats = cleaned[rows]
            valid_s = valid[rows].astype(np.float32)
            conf_s = np.clip(df["confidence"].to_numpy(np.float32)[rows], 0, 1)
            frame_idx = (
                df["frame"].to_numpy(dtype=int)[rows] - 1 if "frame" in df
                else rows.copy()
            )
            frame_idx = np.clip(frame_idx, 0, None)
            boxes = self._head_boxes(df, valid)
            boxes = boxes[rows] if boxes is not None else None
            if boxes is None:
                logger.warning("%s: no head boxes; Gaze-LLE features are zero", video_path)
        else:
            logger.warning("%s: OpenFace returned no rows", video_path)
            of_feats = np.zeros((target_t, len(self.OPENFACE_COLS)), dtype=np.float32)
            valid_s = conf_s = np.zeros(target_t, dtype=np.float32)
            total = self._video_frame_count(video_path)
            frame_idx = (
                np.linspace(0, total - 1, target_t).round().astype(int)
                if total else np.zeros(target_t, dtype=int)
            )
            boxes = None

        parts = [of_feats]
        if self.include_quality_flags:
            parts.append(np.column_stack((valid_s, conf_s)))
        if self.add_pose_delta:
            start = self.OPENFACE_COLS.index(self.POSE[0])
            pose = of_feats[:, start:start + len(self.POSE)]
            parts.append(np.diff(pose, axis=0, prepend=pose[:1]).astype(np.float32))

        tabular = torch.from_numpy(np.concatenate(parts, axis=1).astype(np.float32))
        gaze = self.extract_gaze_features(video_path, frame_idx, boxes)
        features = torch.cat((tabular, gaze), dim=-1).unsqueeze(0)

        if not torch.isfinite(features).all():
            logger.warning("%s: replacing non-finite features with zero", video_path)
            features = torch.nan_to_num(features, nan=0.0, posinf=0.0, neginf=0.0)

        expected = (1, target_t, self.feature_dim)
        if tuple(features.shape) != expected:
            raise ValueError(f"Expected shape {expected}, got {tuple(features.shape)}")
        return features

In [7]:
import  glob, os
import torch
from src.common.constants import OPENFACE_BIN

# Gaze-LLE repo must be importable (provides src.models.gazelle)

# If the class lives in a file instead of an earlier cell:
# from visual_features import VisualFeatureExtractor

extractor = VisualFeatureExtractor(
    gazelle_checkpoint_path="/home/dandy/multimodal-deception-detector-backbone-model/src/models/gazelle_dinov2_vitl14.pt",
    gazelle_model_name="gazelle_dinov2_vitl14",
    openface_bin_path=OPENFACE_BIN,   # or full path to the binary
    batch_size=8,
)

# --- single video ---
video_path = "/path/to/video.mp4"
feats = extractor.extract_features(video_path, target_t=90)
print("shape:", tuple(feats.shape))          # (1, 90, feature_dim)
print("feature_dim:", extractor.feature_dim)

# Sanity check: peek at the Gaze-LLE columns for a few timesteps
names = extractor.feature_names
gz_start = next(i for i, n in enumerate(names) if n.startswith("gz_"))
print(names[gz_start:])
print(feats[0, :5, gz_start:])

# --- optional: batch over a folder and save ---
video_dir = "/path/to/videos"
out_dir = "/path/to/features"
os.makedirs(out_dir, exist_ok=True)

for vp in sorted(glob.glob(os.path.join(video_dir, "*.mp4"))):
    out_path = os.path.join(out_dir, os.path.splitext(os.path.basename(vp))[0] + ".pt")
    if os.path.exists(out_path):
        continue
    try:
        torch.save(extractor.extract_features(vp, target_t=90).squeeze(0).cpu(), out_path)
    except Exception as e:
        print(f"FAILED {vp}: {e}")

Using cache found in /home/dandy/.cache/torch/hub/facebookresearch_dinov2_main


FileNotFoundError: OpenFace CSV not found: /tmp/tmp86wlw4ri/video.csv